In [1]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.alpha.lab import AlphaLab
from vnpy.alpha import logger, WindowGenerator, LGBMLR_Runner
from vnpy.trader.constant import Interval
from datetime import datetime
from pathlib import Path
from vnpy.alpha.strategy import BacktestingEngine2
import vnpy.alpha.strategy.strategies.equity_demo_strategy2 as equity_demo_strategy2
EquityDemoStrategy = equity_demo_strategy2.EquityDemoStrategy2


In [2]:
# ============================================================================
# Cell 2: 配置与初始化
# ============================================================================
vt_index_symbol = "000300.SSE"
BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'

lab = AlphaLab(str(LAB_PATH))
dataset = lab.load_dataset('v100')

n_quantiles = 30

In [3]:
# ============================================================================
# Cell 3: 生成 WalkForward 窗口
# ============================================================================
windows = WindowGenerator.generate(
    start=datetime(2018, 1, 1),
    end=datetime(2026, 5, 8),
    train_years=3.0,
    valid_years=1.0,
    test_years=1.0,
    step_years=1.0,
)

print(f"生成 {len(windows)} 个窗口:")
for w in windows:
    print(
        f"  Win{w.index}: "
        f"train={w.train_start.date()}~{w.train_end.date()}, "
        f"valid={w.valid_start.date()}~{w.valid_end.date()}, "
        f"test={w.test_start.date()}~{w.test_end.date()}"
    )

生成 5 个窗口:
  Win0: train=2018-01-01~2021-01-01, valid=2021-01-01~2022-01-01, test=2022-01-01~2023-01-01
  Win1: train=2019-01-01~2022-01-01, valid=2022-01-01~2023-01-01, test=2023-01-01~2024-01-01
  Win2: train=2020-01-01~2023-01-01, valid=2023-01-01~2024-01-01, test=2024-01-01~2025-01-01
  Win3: train=2021-01-01~2024-01-01, valid=2024-01-01~2025-01-01, test=2025-01-01~2026-01-01
  Win4: train=2022-01-01~2025-01-01, valid=2025-01-01~2026-01-01, test=2026-01-01~2026-05-08


In [4]:
# ============================================================================
# Cell 5: 单次滚动回测（固定超参数）
# ============================================================================
runner = LGBMLR_Runner(
    name='lgb_baseline',
    lab=lab,
    dataset=dataset,
    engine_class=BacktestingEngine2,
    strategy_class=EquityDemoStrategy,
    windows = windows,
    model_params={
        'num_leaves': 512,
        'max_depth': -1,
        'min_data_in_leaf': 140,
        'learning_rate': 0.001,
        'feature_fraction': 0.88,
        'bagging_fraction': 0.87,
        'bagging_freq': 5,
        'lambda_l1': 30,
        'lambda_l2': 0.0,
    },
    strategy_params={
        'top_k': 5,
        'min_days': 6,
        'cash_ratio': 1.0,
        'open_rate': 0.0005,
        'close_rate': 0.0015,
        'min_commission': 5,
        'slippage': 0.0006,
        'price_add': 0.05,
    },
    backtest_params={
        'interval': Interval.DAILY,
        'capital': 100_000,
        'risk_free': 0.0,
        'annual_days': 240,
        'min_commission': 5.0,
        'slippage': 0.0006,
        'adjust_type': 'none',
    },
    benchmark_symbol='000300.SSE',
    n_quantiles=30,
    seed=42,
)

stats = runner.run(runner.windows, log = False)

print(f"训练了 {len(runner.models)} 个模型")
print(f"signals 列表长度（含拼接信号）: {len(runner.signals)}")
print(f"engine 实例: {runner.engine}")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	train's ndcg@5: 0.384256	train's ndcg@10: 0.38694	valid's ndcg@5: 0.347347	valid's ndcg@10: 0.346873
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[282]	train's ndcg@5: 0.470613	train's ndcg@10: 0.455392	valid's ndcg@5: 0.375171	valid's ndcg@10: 0.368281
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[7]	train's ndcg@5: 0.443353	train's ndcg@10: 0.437722	valid's ndcg@5: 0.344489	valid's ndcg@10: 0.342418
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[283]	train's ndcg@5: 0.468577	train's ndcg@10: 0.450174	valid's ndcg@5: 0.406937	valid's ndcg@10: 0.399944
Evaluated only: ndcg@5
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration 

100%|██████████| 405/405 [00:02<00:00, 178.32it/s]


AttributeError: 'Handler' object has no attribute '_level'

In [ ]:
logger.info('123')

In [ ]:
# import polars as pl
# with pl.Config(tbl_rows=317, tbl_cols=None, fmt_str_lengths=None):
#     display(runner.engine.daily_df)